In [38]:
!pip install pandas numpy openpyxl

In [39]:
from smtplib import SMTP_SSL
sender='rajdippharma1989@gmail.com'
recipient = "rajdipforitdeveloper@yahoo.com"
password = "tecj oqqi raek peey"

# Using SMTP_SSL with port 465 is more reliable and prevents unexpected disconnects
connect_server = SMTP_SSL(host = 'smtp.gmail.com', port = 465)
connect_server.login(user = sender, password = password)
connect_server.sendmail(from_addr = sender, to_addrs = recipient, msg = "Hey, There!")
connect_server.close()

In [40]:
from smtplib import SMTP_SSL
sender='rajdipforitdeveloper@yahoo.com'
recipient = "rajdippharma1989@gmail.com"
password = "mwltphuixxnozvrs"

# Constructing an RFC 822 compliant raw email string
# Explicit From, To, and Subject headers are required to bypass Yahoo's spam filters
msg = f"""From: {sender}
To: {recipient}
Subject: Discount Offer

Hey, There!"""

# Connecting to Yahoo's SMTP server
connect_server = SMTP_SSL(host = 'smtp.mail.yahoo.com', port = 465)
connect_server.login(user = sender, password = password)
connect_server.sendmail(from_addr = sender, to_addrs = recipient, msg = msg)
connect_server.close()

In [41]:
#import required modules
from smtplib import SMTP_SSL
import datetime
import pandas as pd

In [42]:
#Assign and declared required variables
Password = "tecj oqqi raek peey"
Sender = 'rajdippharma1989@gmail.com'
now = datetime.datetime.now()
actual_date = (now.day, now.month)

#Read from Excel sheet using pandas module
Emails_list = pd.read_excel('/content/emails_list.xlsx')

#Extract the consumers names and the customers emails from the Excel sheet as a list
names = Emails_list['Name']
emails = Emails_list['Email']

#connect to the server of the sender email account
#Using SMTP_SSL already encrypts the connection on port 465, so starttls() is not needed
with SMTP_SSL(host = 'smtp.gmail.com', port = 465) as connect_server:
  #Log in the sender account (matching the capitalized variable Sender)
  connect_server.login(user = Sender, password = Password)

  #iterate through records to get each name and email
  for item in range(len(emails)):
    name = names[item]
    email = emails[item]

    #Adjust the time of sending emails to the customers using control flow if statement
    if actual_date == (20,11):
      message = "Subject: Black Friday Discount Offer\n\n Hi, {}, Do not wait & buy now.".format(name)
      connect_server.sendmail(Sender, [email], message)
    elif actual_date == (20,12):
      message = "Subject: White Friday Discount Offer\n\n Hi, {}, Do not wait & buy now.".format(name)
      connect_server.sendmail(Sender, [email], message)
    #Testing Our Code By Sending Emails & print Statement
    elif actual_date == (5,12):
      message = "Subject: Discount Offer\n\n Hi, {}, Code Is Okay..".format(name)
      connect_server.sendmail(Sender, [email], message)
      print("Code Testing Result Is Okay.")

print('All Done.')

All Done.


In [43]:

# Install required packages
!pip install -q pandas openpyxl

# Import required modules
import pandas as pd
import datetime
import smtplib

from email.message import EmailMessage
from google.colab import userdata

# --------------------------------------------------
# 1. EMAIL ACCOUNT CONFIGURATION
# --------------------------------------------------

try:
    Sender = userdata.get("GMAIL_SENDER")
    Password = userdata.get("GMAIL_APP_PASSWORD")
except Exception:
    # Fallback to local variables if Secrets are not configured in Colab sidebar
    Sender = 'rajdippharma1989@gmail.com'
    Password = 'tecj oqqi raek peey'

SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 465

# Keep True while testing.
# Change to False only when ready to send real emails.
DRY_RUN = True

# --------------------------------------------------
# 2. DATE CONFIGURATION
# --------------------------------------------------

today = datetime.date.today()

print("Current Date:", today.strftime("%d-%m-%Y"))

# Campaign configuration
campaigns = {
    (20, 11): {
        "subject": "Black Friday Discount Offer",
        "body": "Do not wait! Grab your Black Friday discount offer now."
    },

    (20, 12): {
        "subject": "White Friday Discount Offer",
        "body": "Do not wait! Enjoy our special White Friday offer."
    },

    (5, 12): {
        "subject": "Discount Offer",
        "body": "Your discount offer is available now."
    }
}

# Find the campaign for today's date
actual_date = (today.day, today.month)

campaign = campaigns.get(actual_date)

if campaign is None:
    print("No email campaign is scheduled for today.")

else:

    # --------------------------------------------------
    # 3. READ EXCEL CUSTOMER DATA
    # --------------------------------------------------

    Emails_list = pd.read_excel("/content/emails_list.xlsx")

    # Validate required columns
    required_columns = ["Name", "Email"]

    missing_columns = [
        column for column in required_columns
        if column not in Emails_list.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing Excel columns: {missing_columns}"
        )

    # Remove records with missing information
    Emails_list = Emails_list.dropna(
        subset=["Name", "Email"]
    ).copy()

    # Clean customer data
    Emails_list["Name"] = Emails_list["Name"].astype(str).str.strip()
    Emails_list["Email"] = Emails_list["Email"].astype(str).str.strip()

    # Remove duplicate email addresses
    Emails_list = Emails_list.drop_duplicates(
        subset=["Email"]
    )

    # --------------------------------------------------
    # 4. DISPLAY CAMPAIGN INFORMATION
    # --------------------------------------------------

    print("\nCampaign Selected:", campaign["subject"])
    print("Total Customers:", len(Emails_list))

    # --------------------------------------------------
    # 5. CONNECT AND SEND EMAILS
    # --------------------------------------------------

    successful_emails = 0
    failed_emails = 0

    if DRY_RUN:

        print("\nDRY RUN MODE: No actual emails will be sent.")

        for row in Emails_list.itertuples(index=False):

            print(
                f"READY: {row.Name} -> {row.Email}"
            )

        print("\nDry run completed successfully.")

    else:

        if not Sender or not Password:
            raise ValueError("Email credentials are missing.")

        with smtplib.SMTP_SSL(
            SMTP_HOST,
            SMTP_PORT,
            timeout=30
        ) as connect_server:

            # Authenticate sender account
            connect_server.login(Sender, Password)

            # Iterate through customer records
            for row in Emails_list.itertuples(index=False):

                try:

                    message = EmailMessage()

                    message["From"] = Sender
                    message["To"] = row.Email
                    message["Subject"] = campaign["subject"]

                    message.set_content(
                        f"Hi {row.Name},\n\n"
                        f"{campaign['body']}\n\n"
                        "Thank you."
                    )

                    connect_server.send_message(message)

                    successful_emails += 1

                    print(
                        f"SUCCESS: Email sent to {row.Email}"
                    )

                except Exception as error:

                    failed_emails += 1

                    print(
                        f"FAILED: {row.Email} | {error}"
                    )

    # --------------------------------------------------
    # 6. FINAL EXECUTION SUMMARY
    # --------------------------------------------------

    print("\n========== EMAIL EXECUTION SUMMARY ==========")

    print("Campaign:", campaign["subject"])
    print("Total Customers:", len(Emails_list))
    print("Successfully Sent:", successful_emails)
    print("Failed:", failed_emails)
    print("Dry Run:", DRY_RUN)

    if DRY_RUN:
        print("Status: TEST COMPLETED. No emails were sent.")

    elif failed_emails == 0:
        print("Status: ALL EMAILS SENT SUCCESSFULLY.")

    else:
        print("Status: COMPLETED WITH SOME FAILURES.")

Current Date: 18-11-2026
No email campaign is scheduled for today.


In [44]:

# ==========================================================
# BUSINESS EMAIL LIST - EMAIL MARKETING AUTOMATION
# Python | Pandas | SMTP | Excel | Google Colab
# ==========================================================

!pip install -q pandas openpyxl

# 1. IMPORT REQUIRED LIBRARIES
import pandas as pd
import datetime
import smtplib
import re

from email.message import EmailMessage
from google.colab import userdata
from pathlib import Path

# ==========================================================
# 2. CONFIGURATION
# ==========================================================

SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 465

# True = Testing, no emails sent
# False = Send actual emails
DRY_RUN = True

# Testing date: Change to None for actual current date
TEST_DATE = datetime.date(2026, 11, 20)

EXCEL_FILE = "/content/emails_list.xlsx"

# ==========================================================
# 3. DATE AND CAMPAIGN CONFIGURATION
# ==========================================================

today = TEST_DATE if TEST_DATE else datetime.date.today()

print("Current Execution Date:", today.strftime("%d-%m-%Y"))

campaigns = {
    (20, 11): {
        "subject": "Black Friday Discount Offer",
        "body": "Do not wait! Grab your Black Friday discount offer now."
    },

    (20, 12): {
        "subject": "White Friday Discount Offer",
        "body": "Do not wait! Enjoy our special White Friday offer."
    },

    (5, 12): {
        "subject": "Discount Offer",
        "body": "Your discount offer is available now."
    }
}

actual_date = (today.day, today.month)
campaign = campaigns.get(actual_date)

# ==========================================================
# 4. INITIALIZE EXECUTION COUNTERS
# ==========================================================

total_customers = 0
prepared_emails = 0
successful_emails = 0
failed_emails = 0

# ==========================================================
# 5. CHECK CAMPAIGN AVAILABILITY
# ==========================================================

if campaign is None:

    print("\nNo email campaign is scheduled for this date.")

else:

    print("\nCampaign Selected:", campaign["subject"])

    # ======================================================
    # 6. READ EXCEL CUSTOMER DATA
    # ======================================================

    if not Path(EXCEL_FILE).is_file():
        raise FileNotFoundError(
            f"Excel file not found: {EXCEL_FILE}"
        )

    Emails_list = pd.read_excel(EXCEL_FILE)

    # Remove unnecessary spaces from column names
    Emails_list.columns = Emails_list.columns.astype(str).str.strip()

    required_columns = ["Name", "Email"]

    missing_columns = [
        col for col in required_columns
        if col not in Emails_list.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # Select only required columns
    Emails_list = Emails_list[["Name", "Email"]].copy()

    # Remove missing values
    Emails_list = Emails_list.dropna(
        subset=["Name", "Email"]
    )

    # Clean customer records
    Emails_list["Name"] = (
        Emails_list["Name"].astype(str).str.strip()
    )

    Emails_list["Email"] = (
        Emails_list["Email"].astype(str).str.strip().str.lower()
    )

    # Remove empty records
    Emails_list = Emails_list[
        (Emails_list["Name"] != "") &
        (Emails_list["Email"] != "")
    ]

    # Remove duplicate email addresses
    Emails_list = Emails_list.drop_duplicates(
        subset=["Email"]
    )

    # Validate email addresses
    email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

    valid_mask = Emails_list["Email"].str.match(
        email_pattern, na=False
    )

    invalid_emails = Emails_list[~valid_mask]

    Emails_list = Emails_list[valid_mask].copy()

    total_customers = len(Emails_list)

    print("\nTotal Valid Customers:", total_customers)
    print("Invalid Email Records:", len(invalid_emails))

    if total_customers == 0:
        print("\nNo valid customer records found.")

    else:

        # ==================================================
        # 7. PREPARE PERSONALIZED EMAILS
        # ==================================================

        email_messages = []

        for row in Emails_list.itertuples(index=False):

            message = EmailMessage()

            message["From"] = (
                userdata.get("GMAIL_SENDER")
                if not DRY_RUN
                else "test@example.com"
            )

            message["To"] = row.Email
            message["Subject"] = campaign["subject"]

            message.set_content(
                f"Hi {row.Name},\n\n"
                f"{campaign['body']}\n\n"
                "Thank you."
            )

            email_messages.append(message)

            prepared_emails += 1

        print("\nPrepared Emails:", prepared_emails)

        # ==================================================
        # 8. DRY RUN OR ACTUAL EMAIL EXECUTION
        # ==================================================

        if DRY_RUN:

            print("\nDRY RUN MODE ENABLED")
            print("No actual emails will be sent.\n")

            for message in email_messages:

                print(
                    f"READY: {message['To']} | "
                    f"{message['Subject']}"
                )

        else:

            # Retrieve credentials from Colab Secrets
            Sender = userdata.get("GMAIL_SENDER")
            Password = userdata.get("GMAIL_APP_PASSWORD")

            if not Sender or not Password:
                raise ValueError(
                    "Gmail credentials are missing from Colab Secrets."
                )

            # Connect to Gmail SMTP server
            with smtplib.SMTP_SSL(
                SMTP_HOST,
                SMTP_PORT,
                timeout=30
            ) as connect_server:

                connect_server.login(Sender, Password)

                # Send individual emails
                for message in email_messages:

                    message.replace_header("From", Sender)

                    try:

                        refused = connect_server.send_message(message)

                        if refused:
                            failed_emails += 1
                            print(
                                f"FAILED: {message['To']} | "
                                f"Recipient rejected."
                            )

                        else:
                            successful_emails += 1

                            print(
                                f"SUCCESS: {message['To']}"
                            )

                    except Exception as error:

                        failed_emails += 1

                        print(
                            f"FAILED: {message['To']} | {error}"
                        )

# ==========================================================
# 9. FINAL EXECUTION SUMMARY
# ==========================================================

print("\n" + "=" * 48)
print("       EMAIL EXECUTION SUMMARY")
print("=" * 48)

print("Execution Date:", today.strftime("%d-%m-%Y"))
print("Campaign:", campaign["subject"] if campaign else "None")
print("Total Valid Customers:", total_customers)
print("Prepared Emails:", prepared_emails)
print("Successfully Sent:", successful_emails)
print("Failed Emails:", failed_emails)
print("Dry Run Mode:", DRY_RUN)

if campaign is None:
    print("STATUS: NO SCHEDULED CAMPAIGN")

elif DRY_RUN:
    print("STATUS: TEST COMPLETED - NO EMAILS SENT")

elif failed_emails == 0:
    print("STATUS: ALL EMAILS SENT SUCCESSFULLY")

else:
    print("STATUS: COMPLETED WITH SOME FAILURES")

print("=" * 48)
print("Execution Finished.")


Current Execution Date: 20-11-2026

Campaign Selected: Black Friday Discount Offer

Total Valid Customers: 3
Invalid Email Records: 0

Prepared Emails: 3

DRY RUN MODE ENABLED
No actual emails will be sent.

READY: rajdiptheitdeveloper@yahoo.com | Black Friday Discount Offer
READY: rajdipforitdeveloper@yahoo.com | Black Friday Discount Offer
READY: rajdippharma1989@gmail.com | Black Friday Discount Offer

       EMAIL EXECUTION SUMMARY
Execution Date: 20-11-2026
Campaign: Black Friday Discount Offer
Total Valid Customers: 3
Prepared Emails: 3
Successfully Sent: 0
Failed Emails: 0
Dry Run Mode: True
STATUS: TEST COMPLETED - NO EMAILS SENT
Execution Finished.
